# CBB Team Clustering Models

### Goal: Use K-means and K-nearest neighbors models to compare teams and identify team archetypes

## Imports and function definitions

In [16]:
import os
from google.cloud import bigquery
import pandas as pd
import db_dtypes
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA

In [17]:
import sys
from pathlib import Path

project_root=Path.cwd().parent 
sys.path.append(str(project_root))

In [18]:
from src.utils.scaler import scale_features

In [19]:
from src.visualization.scatterplots import plot_archetypes

In [20]:
from src.models.knn_similarity import generate_comps_table

In [21]:
from src.utils.map_stat_labels import map_stat_labels

In [22]:
from src.visualization.radar_plots import draw_radar_plot


## Load data 

In [23]:
from src.data.load_data import load_data
stats=load_data()[0]
stat_dict=load_data()[1]

## Scale Features
### We use a standard scaler and apply it to all metrics in the table for better interpretation from the model

In [24]:

# scale features grouped by each season for normalization based on season context
metrics=['stlPct','blkPct','pfPct','atr2FgaFreq', 'atr2FgPct','freqChncTr', 'paint2FgPct','paint2FgaFreq' ,'mid2FgPct','mid2FgaFreq','fga3Rate', 'orbPct', 'drbPct', 'astPct','ortg','drtg' ,'pace', 'ftaRate', 'tovPct','fg3Pct','lane2FgPctAgst','lane2FgaFreqAgst', 'mid2FgPctAgst','mid2FgaFreqAgst','fga3RateAgst','fg3PctAgst']

stats_scaled=stats.groupby('competitionId',group_keys=False).apply(scale_features, columns=metrics)

/var/folders/n4/3pnvvzbj0hl6zmtjl31jyk4c0000gn/T/ipykernel_45407/2590478368.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats_scaled=stats.groupby('competitionId',group_keys=False).apply(scale_features, columns=metrics)


In [25]:
## create a dataset with percentile ranks for each team relative to each season
rank = (
    stats.drop(columns=['teamId','competitionId','teamMarket','teamName','competitionName','hexColor1','hexColor2'])
         .groupby(stats['competitionId'])
         .rank(pct=True)
)

rank = pd.concat(
    [stats[['teamId','competitionId','teamMarket','teamName','competitionName','hexColor1','hexColor2']], rank],
    axis=1
)


## Create and train KNN model

In [36]:
example=generate_comps_table(team='Michigan',season=41097,metrics=metrics,rank=rank,stats_scaled=stats_scaled,same_season=False)

## Shooting Style KMeans model

In [27]:
metrics=['atr2FgaFreq', 'atr2FgPct', 'paint2FgPct','paint2FgaFreq' ,'mid2FgPct','mid2FgaFreq','fga3Rate', 'fg3Pct']

Y=stats_scaled[metrics]
kmeans = KMeans(n_clusters=4,random_state=3)
labels = kmeans.fit_predict(Y)
rank['arch']=labels

cluster_mean_percentiles=rank.groupby('arch')[metrics].mean(numeric_only=True).reset_index().set_index('arch')

cluster_names={0: 'Paint Dominant',1:'Rim and Three',2:'Elite Shooting',3: 'Midrange Heavy'}
rank['shooting_archetype']=rank['arch'].map(cluster_names)

In [28]:
## Perform PCA for dimensionality reduction and visualization
pca=PCA(n_components=2)
coords=pca.fit_transform(Y)


rank['style_x']=coords[:,0]
rank['style_y']=coords[:,1]

def top_features(loadings, features, n=2):
    idx = np.argsort(np.abs(loadings))[::-1]
    return [(features[i], loadings[i]) for i in idx[:n]]

print(top_features(pca.components_[0],metrics))
print(top_features(pca.components_[1],metrics))


[('fga3Rate', np.float64(0.5589197201011148)), ('mid2FgaFreq', np.float64(-0.5088414827965495))]
[('paint2FgaFreq', np.float64(0.49259589460482484)), ('atr2FgaFreq', np.float64(-0.47942345102171324))]


In [29]:
# plot to visualize shooting archetype clusters
plot_archetypes(df=rank,x_col='style_x',y_col='style_y',color_col='shooting_archetype',hover_cols=['teamMarket','competitionName'],x_title='<- mid-range heavy | 3PT heavy ->',y_title='<- paint heavy | rim heavy ->')

In [30]:
# Example comparison table for shooting metrics
shooting_metrics=['atr2FgaFreq', 'atr2FgPct', 'paint2FgPct','paint2FgaFreq' ,'mid2FgPct','mid2FgaFreq','fga3Rate', 'fg3Pct']
shooting_example=generate_comps_table('Duke',41097,shooting_metrics,same_season=False,rank=rank,stats_scaled=stats_scaled)
shooting_example[:3]

,Team,similarity,hexColor1,hexColor2,atr2FgaFreq,atr2FgPct,paint2FgPct,paint2FgaFreq,mid2FgPct,mid2FgaFreq,fga3Rate,fg3Pct
0,Duke 2025-26,1.000000,#1c4c9c,#204c9c,0.852055,0.882192,0.864384,0.263014,0.852055,0.054795,0.734247,0.608219
1,Utah St. 2024-25,0.963706,#ada093,#162535,0.868132,0.755495,0.898352,0.351648,0.817308,0.056319,0.747253,0.782967
2,Southern Utah 2020-21,0.947246,#2d2229,#c92424,0.829971,0.850144,0.861671,0.256484,0.582133,0.100865,0.835735,0.561960


In [31]:
# Draw a radar plot for shooting metrics comparison
draw_radar_plot(shooting_example,shooting_metrics,stat_dict)

## Playstyle KMeans model

In [32]:

playstyle_metrics=['astPct','tovPct','pace','orbPct','ftaRate']

Y=stats_scaled[playstyle_metrics]

kmeans = KMeans(n_clusters=4,random_state=0)
labels = kmeans.fit_predict(Y)
rank['playstyle_arch']=labels

rank.groupby('playstyle_arch')[playstyle_metrics].mean(numeric_only=True)

# 0-Controlled Ball Movement-high assists, low turnovers, slow pace
# 1- High-Tempo aggressive- high pace, high turnovers, high rebound/ft
# 2-Inefficient isolation- very high turnovers, low rebounding and assists
# 3- Slow isolation and physical- hiigh rebounding and slow pace

playstyle_archetypes={0:'Controlled Ball Movement',1:'High-Tempo Aggressive',2:'Inefficient Isolation',3:'Slow and Physical'}
rank['playstyle_arch_name']=rank['playstyle_arch'].map(playstyle_archetypes)

In [33]:
pca=PCA(n_components=2)
coords=pca.fit_transform(Y)

rank['p_style_x']=coords[:,0]
rank['p_style_y']=coords[:,1]

print(top_features(pca.components_[0],playstyle_metrics))
print(top_features(pca.components_[1],playstyle_metrics))


[('ftaRate', np.float64(0.6798621214387162)), ('pace', np.float64(0.46065303073203057))]
[('tovPct', np.float64(0.6045031534195096)), ('orbPct', np.float64(-0.5472775853063141))]


In [34]:
plot_archetypes(rank,x_col='p_style_x',y_col='p_style_y',color_col='playstyle_arch_name',hover_cols=['teamMarket','competitionName'],x_title='<- Slow and Controlled  | Fast and Aggressive ->',y_title='<- Physical Rebounding | High Turnovers ->',title='Team Playstyle Archetypes')

In [37]:
playstyle_example=generate_comps_table('Duke',41097,playstyle_metrics,rank=rank,stats_scaled=stats_scaled)
playstyle_example[:3]

,Team,similarity,hexColor1,hexColor2,astPct,tovPct,pace,orbPct,ftaRate
0,Duke 2025-26,1.000000,#1c4c9c,#204c9c,0.865753,0.313699,0.282192,0.983562,0.764384
1,Purdue 2022-23,0.972309,#20201f,#c8b487,0.966942,0.212121,0.060606,0.994490,0.900826
2,Southern California 2021-22,0.965127,#fac01c,#a41c33,0.659218,0.343575,0.370112,0.896648,0.586592


In [39]:
draw_radar_plot(playstyle_example,playstyle_metrics,stat_dict)

## Defense Style KMeans model

In [40]:
defense_metrics=['blkPct','stlPct','drbPct','pfPct','tovPctAgst','lane2FgPctAgst','lane2FgaFreqAgst','fga3RateAgst','fg3PctAgst']

Y=stats_scaled[defense_metrics]

kmeans = KMeans(n_clusters=4,random_state=1)
labels = kmeans.fit_predict(Y)
rank['defense_arch']=labels

rank.groupby('defense_arch')[defense_metrics].median(numeric_only=True)

# 0-Disruptive Rim Protection: low lane FG% and high freq, low 3PA allowed (protects rim well but allows drives)
# 2-Soft defense-Very high paint fg% allowed and 3pt pct allowed
# 3-paint containment defense-Low rim freq, high blocks, very high 3pt rate (Prevents rim attempts)
# 4-conservative rebounding defense-Low blocks steals and fouls, high rebounding

defensive_archetype_names={0:'Disruptive Rim Protection',1:'Soft Defense',2:'Paint Containment Defense',3:'Conservative Rebounding Defense'}
rank['defense_arch_name']=rank['defense_arch'].map(defensive_archetype_names)

In [41]:
pca=PCA(n_components=2)
coords=pca.fit_transform(Y)

rank['d_style_x']=coords[:,0]
rank['d_style_y']=coords[:,1]

print(top_features(pca.components_[0],defense_metrics))
print(top_features(pca.components_[1],defense_metrics))

[('fga3RateAgst', np.float64(0.6932133802616175)), ('lane2FgaFreqAgst', np.float64(-0.6832657574791956))]
[('blkPct', np.float64(0.6032518231183911)), ('lane2FgPctAgst', np.float64(-0.5932880885449557))]


In [42]:
plot_archetypes(rank,x_col='d_style_x',y_col='d_style_y',color_col='defense_arch_name',hover_cols=['teamMarket','competitionName'],x_title='<- Forces Paint Shots | Forces 3-pointers ->',y_title='<- Low Blocks and Rebounds | High Blocks and Rebounds ->',title='Team Defensive Archetypes')

In [47]:
defense_example=generate_comps_table('Duke',41097,defense_metrics,rank,stats_scaled)
defense_example[:3]

,Team,similarity,hexColor1,hexColor2,blkPct,stlPct,drbPct,pfPct,tovPctAgst,lane2FgPctAgst,lane2FgaFreqAgst,fga3RateAgst,fg3PctAgst
0,Duke 2025-26,1.000000,#1c4c9c,#204c9c,0.676712,0.758904,0.964384,0.178082,0.608219,0.084932,0.035616,0.920548,0.093151
1,Gonzaga 2025-26,0.967282,#babac5,#22305f,0.756164,0.895890,0.939726,0.219178,0.920548,0.106849,0.101370,0.794521,0.076712
2,Clemson 2025-26,0.907777,#f4541c,#f8541c,0.454795,0.484932,0.912329,0.397260,0.656164,0.246575,0.149315,0.830137,0.279452


In [49]:
draw_radar_plot(defense_example,defense_metrics,stat_dict)